# Building and importing Python Libraries

In the previous chapter we tried to setup an old webserver - a common example of an enterprise project that has grown over time but where the development in terms of technologies and libraries didn't followed. Due tue time pressure and capacity, these projects stayed untouched - they run in production.

However, Python gives us the opportunity to import functions from other projects. Distributed as ``pypi`` packages, they can easily be imported into existing projects. 

In the following we are going to write an own library called ``PyGuard`` and we try to import it into Bob's server to secure some of the offered endpoints. 

## Implement the middleware

---

## Setup Bob's server

At first, we are setting up Bob's server (once again)

In [5]:
!rm -rf $HOME/tmp/pyguard_demo && mkdir -p $HOME/tmp/pyguard_demo

In [6]:
!cp -r /home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/projXY_bobs_webserver $HOME/tmp/pyguard_demo

In [7]:
!ls -la $HOME/tmp/pyguard_demo/projXY_bobs_webserver

total 1656
drwxr-xr-x 6 fixcfhu fixcfhu    4096 Jul 30 07:45 .
drwxr-xr-x 3 fixcfhu fixcfhu    4096 Jul 30 07:45 ..
drwxr-xr-x 2 fixcfhu fixcfhu    4096 Jul 30 07:45 .ipynb_checkpoints
-rw-r--r-- 1 fixcfhu fixcfhu    1189 Jul 30 07:45 Dockerfile
-rw-r--r-- 1 fixcfhu fixcfhu     912 Jul 30 07:45 README.md
drwxr-xr-x 3 fixcfhu fixcfhu    4096 Jul 30 07:45 etc
-rw-r--r-- 1 fixcfhu fixcfhu 1644552 Jul 30 07:45 image.png
-rw-r--r-- 1 fixcfhu fixcfhu   13233 Jul 30 07:45 legacy_python_projects.ipynb
drwxr-xr-x 2 fixcfhu fixcfhu    4096 Jul 30 07:45 secrets
drwxr-xr-x 4 fixcfhu fixcfhu    4096 Jul 30 07:45 server


In [8]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && export UV_NATIVE_TLS=1 && uv python install 3.9 && uv python pin 3.9 && uv venv --clear

Pinned `.python-version` to `3.9`
Using CPython 3.9.23
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [9]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m ensurepip

Looking in links: /tmp/tmpr7_ucwtu
Processing /tmp/tmpr7_ucwtu/setuptools-58.1.0-py3-none-any.whl
Processing /tmp/tmpr7_ucwtu/pip-23.0.1-py3-none-any.whl


In [10]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip install -r server/requirements.txt

  Using cached requests-2.0.0-py2.py3-none-any.whl (391 kB)
  Using cached urllib3-1.7.1.tar.gz (67 kB)
  Preparing metadata (setup.py) ... done
  Using cached certifi-2015.04.28-py2.py3-none-any.whl (373 kB)
  Using cached chardet-2.1.1.tar.gz (178 kB)
  Preparing metadata (setup.py) ... done
  DEPRECATION: urllib3 is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559
  Running setup.py install for urllib3 ... done
  DEPRECATION: chardet is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at htt

In [11]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip freeze

certifi==2015.4.28
chardet==2.1.1
requests==2.0.0
urllib3==1.7.1


In [ ]:
!tree -L 2 -a $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9

In [30]:
# a small test if the server is up and running
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 server/main.py

/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/server/main.py:10: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  import imp
Using modern compatibility mode.
Legacy API Service
Python: 3.9.23 (main, Sep  2 2025, 14:19:32) 
[Clang 20.1.4 ]
Server: legacy-api
Build: 1837
Compatibility: 7
Listening on http://127.0.0.1:8000

^C
Traceback (most recent call last):
  File "/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/server/main.py", line 319, in <module>
    server.serve_forever()
  File "/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/socketserver.py", line 232, in serve_forever
    ready = selector.select(poll_interval)
  File "/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/selectors.py", line 416, in select
    fd_event_list = self._selector.poll(timeout)
KeyboardInterrupt


We are opening a new terminal and see if we can reach the download endpoint of the server
```bash
curl http://127.0.0.1:8000/download?file=api-example.json
curl http://127.0.0.1:8000/download?file=../../secrets/database.conf
```

---

## Import PyGuard

Now it is time, to integrate our ``PyGuard`` middleware into Bob's server. We have seen, that especially the ``download`` middleware bears an attacking surface in terms of path traversal which is a good integration point for ``PyGuard``. 

### Adding it as a editable dependency

When you are working on your Python library locally and you try to integrate it with the target project, you literally want to avoid long development cycles, including the implementation, packaging, uploading, importing etc. because they take time which you do not have. Therefore, Python gives us the opportunity to install libraries as ``editable`` dependencies. In that case, the package manager creates an dynamic link into the source code repository of your library. 

In [12]:
# copying PyGuard into our temp folder
!cp -r /home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/projXY2_pyguard $HOME/tmp/pyguard_demo

In [13]:
# lets get an overview about the current PyGuard folder
!tree -a $HOME/tmp/pyguard_demo/projXY2_pyguard

/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
├── .ipynb_checkpoints
│   └── instructions-checkpoint.ipynb
├── README.md
├── instructions.ipynb
├── pyproject.toml
└── src
    └── pyguard
        ├── __init__.py
        ├── exceptions.py
        ├── middleware.py
        ├── models.py
        ├── rules.py
        └── scanner.py

4 directories, 10 files


In [14]:
# now we are going to install pyguard as a editable dependency
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip install -e $HOME/tmp/pyguard_demo/projXY2_pyguard

Obtaining file:///home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pyguard (pyproject.toml) ... done
  Created wheel for pyguard: filename=pyguard-0.1.0-0.editable-py3-none-any.whl size=1187 sha256=39b04eef8260bb3681f62baa3c843c9bb11905b85ef694ff52e41c981d8f184f
  Stored in directory: /tmp/pip-ephem-wheel-cache-s1cvdzob/wheels/97/27/ee/457ce7c4bf4a1e3cb5e7e8a1defdf7d3cc36405df9e9641333
Successfully built pyguard

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [16]:
# let's check the installed libraries of the virtual environment
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip freeze

certifi==2015.4.28
chardet==2.1.1
# Editable install with no version control (pyguard==0.1.0)
-e /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
requests==2.0.0
urllib3==1.7.1


In [30]:
!tree -L 2 -a $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9

/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9
└── site-packages
    ├── __editable__.pyguard-0.1.0.pth
    ├── __pycache__
    ├── _distutils_hack
    ├── _virtualenv.pth
    ├── _virtualenv.py
    ├── certifi
    ├── certifi-2015.04.28.dist-info
    ├── chardet
    ├── chardet-2.1.1-py3.9.egg-info
    ├── distutils-precedence.pth
    ├── dummyserver
    ├── pip
    ├── pip-23.0.1.dist-info
    ├── pkg_resources
    ├── pyguard-0.1.0.dist-info
    ├── requests
    ├── requests-2.0.0.dist-info
    ├── setuptools
    ├── setuptools-58.1.0.dist-info
    ├── urllib3
    └── urllib3-1.7.1-py3.9.egg-info

19 directories, 4 files


In [31]:
!ls $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages/pyguard-0.1.0.dist-info

INSTALLER  METADATA  RECORD  REQUESTED	WHEEL  direct_url.json	top_level.txt


In [29]:
!cat $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages/pyguard-0.1.0.dist-info/direct_url.json

{"dir_info": {"editable": true}, "url": "file:///home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard"}

In [35]:
!cat $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages/__editable__.pyguard-0.1.0.pth

/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src


In [26]:
!tree -a $HOME/tmp/pyguard_demo/projXY2_pyguard

/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
├── .ipynb_checkpoints
│   └── instructions-checkpoint.ipynb
├── README.md
├── instructions.ipynb
├── pyproject.toml
└── src
    ├── pyguard
    │   ├── __init__.py
    │   ├── __pycache__
    │   │   ├── __init__.cpython-39.pyc
    │   │   ├── exceptions.cpython-39.pyc
    │   │   ├── middleware.cpython-39.pyc
    │   │   ├── models.cpython-39.pyc
    │   │   ├── rules.cpython-39.pyc
    │   │   └── scanner.cpython-39.pyc
    │   ├── exceptions.py
    │   ├── middleware.py
    │   ├── models.py
    │   ├── rules.py
    │   └── scanner.py
    └── pyguard.egg-info
        ├── PKG-INFO
        ├── SOURCES.txt
        ├── dependency_links.txt
        └── top_level.txt

6 directories, 20 files


#### Conclusion

Using the ``-e`` flag we are able to list ``PyGuard`` as an editable dependency within the virtual environment. We have seen that this command creates two artifacts inside the ``site-packages`` of the virtual environment:
* a ``__editable__.pyguard-0.1.0.pth`` which seems to be a symbolic link into our ``PyGuard`` implementation folder
* a ``pyguard-0.1.0.dist-info`` folder with a ``direct_url.json`` giving a hint about the editable dependency and the same file link as the ``.pth`` file

And moreover a ``pyguard.egg-info`` folder in our src code folder. In a next step we want to play with the actual imports a bit more around to familiarize ourselves with the overall concept.

In [32]:
!python3 -c "import pyguard"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'pyguard'


In [33]:
# now let's try to import it from the venv interpreter
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard; print(\"done\")"

done


The ``PyGuard`` import is only valid within the virtual environment, in a next step we need to find our the logic behind the import, therefore we are taking usage of the ``sys`` module which is a package in Python's standard library.

With the ``sys`` lib we can print the ``PYTHONPATH`` variable, which is a list of directories that the Python interpreter is using to lookup library imports.

In [34]:
!python3 -c "import sys; print(sys.path)"

['', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python39.zip', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/lib-dynload', '/home/fixcfhu/.cache/uv/archive-v0/lKpTXibMan_Mf-8uSVQr8/lib/python3.9/site-packages']


In [35]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import sys; print(sys.path)"

['', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python39.zip', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/lib-dynload', '/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages', '/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src']


In [36]:
# We have seen, that the source code folder ``/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src`` is part 
# of the ``PYTHONPATH`` - what happens if we rename it?
!mv /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard_new

In [37]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard; print(\"done\")"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'pyguard'


In [38]:
!mv /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard_new /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard

In [39]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard; print(\"done\")"

done


In [40]:
!cat /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/pyproject.toml

[project]
name = "pyguard"
version = "0.1.0"
requires-python = ">=3.9"

#### Conclusion

The import name of the library is strongly connected to the folder name within the ``src`` folder. This comes from the configuration inside the ``pyproject.toml``.

---

### Integrating and testing the middleware

the following snippets are integrated into the ``main.py``

```python
from pyguard import PyGuardMiddleware
from pyguard.models import Request as GuardRequest
from pyguard.exceptions import RequestBlocked

guard = PyGuardMiddleware()

...

guard_request = GuardRequest(method="GET",path=self.path,)
guard.before_request(guard_request)

except RequestBlocked:
    ...
```

curl http://127.0.0.1:8000/download?file=api-example.json
curl http://127.0.0.1:8000/download?file=../../secrets/database.conf

### Removing the editable dependency

In [37]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip uninstall pyguard -y

Found existing installation: pyguard 0.1.0
Uninstalling pyguard-0.1.0:
  Successfully uninstalled pyguard-0.1.0


In [38]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip freeze

certifi==2015.4.28
chardet==2.1.1
requests==2.0.0
urllib3==1.7.1


In [39]:
!tree -L 2 -a $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9

/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9
└── site-packages
    ├── __pycache__
    ├── _distutils_hack
    ├── _virtualenv.pth
    ├── _virtualenv.py
    ├── certifi
    ├── certifi-2015.04.28.dist-info
    ├── chardet
    ├── chardet-2.1.1-py3.9.egg-info
    ├── distutils-precedence.pth
    ├── dummyserver
    ├── pip
    ├── pip-23.0.1.dist-info
    ├── pkg_resources
    ├── requests
    ├── requests-2.0.0.dist-info
    ├── setuptools
    ├── setuptools-58.1.0.dist-info
    ├── urllib3
    └── urllib3-1.7.1-py3.9.egg-info

18 directories, 3 files


In [40]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import sys; print(sys.path)"

['', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python39.zip', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/lib-dynload', '/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages']
